In [1]:
# ── Preprocessing config — all decisions in one place ─────────────────────────
PKL_PATH = "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_base.pkl"

WINDOW_SIZE         = 7      # days in input window
SKIP_THRESHOLD      = 0.30   # fraction of core features missing per station
MIN_BAD_STATIONS    = 3      # flag a day if this many stations exceed threshold
CORE_FEATURES       = ['TG', 'TN', 'TX', 'UG', 'RH', 'FG', 'Q']  # used for flagging only

In [ ]:
import pickle
import numpy as np

# ── Full path to your pkl file ────────────────────────────────────────────────
PKL_PATH = "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_base.pkl"

with open(PKL_PATH, "rb") as f:
    prepared = pickle.load(f)

print("Keys:", list(prepared.keys()))

data_array = prepared["data_array"]
print("\ndata_array")
print("  shape :", data_array.shape)
print("  dtype :", data_array.dtype)

T, N, F = data_array.shape

dates = prepared["dates"]
print("\ndates")
print("  length:", len(dates))
print("  first :", dates[0])
print("  last  :", dates[-1])

stations = prepared["stations"]
print("\nstations")
print("  shape :", stations.shape)
print(stations.head())

feature_cols = prepared["feature_cols"]
print("\nfeature_cols")
print("  length:", len(feature_cols))
print("  list  :", feature_cols)

rh_idx = prepared["rh_idx"]
print("\nrh_idx :", rh_idx)
print("  name  :", feature_cols[rh_idx])

edge_index = prepared["edge_index"]
print("\nedge_index")
print("  shape :", edge_index.shape)

edge_attr = prepared["edge_attr"]
print("\nedge_attr")
print("  shape :", edge_attr.shape)
print("  min   :", edge_attr.min().round(4))
print("  max   :", edge_attr.max().round(4))

nan_fraction = np.isnan(data_array).mean()
print(f"\nOverall NaN fraction: {nan_fraction:.1%}")

Keys: ['data_array', 'dates', 'stations', 'feature_cols', 'scalers', 'rh_idx', 'edge_index', 'edge_attr', 'K']

data_array
  shape : (10897, 34, 46)
  dtype : float32

dates
  length: 10897
  first : 1995-01-01 00:00:00
  last  : 2024-10-31 00:00:00

stations
  shape : (34, 5)
   node_id           station_name      lat     lon  height_m
0        0  AMSTERDAM/SCHIPHOL AP  52.3172  4.7897     -3.87
1        1              ARCEN AWS  51.4972  6.1961     19.50
2        2           BERKHOUT AWS  52.6428  4.9789     -2.40
3        3       CABAUW TOWER AWS  51.9692  4.9258     -0.71
4        4            DE BILT AWS  52.0989  5.1797      1.90

feature_cols
  length: 46
  list  : ['TG', 'TN', 'TNH', 'TX', 'TXH', 'T10N', 'T10NH', 'UG', 'UN', 'UNH', 'UX', 'UXH', 'EEG', 'RH', 'RHX', 'RHXH', 'DR', 'FG', 'DDVEC', 'FHVEC', 'FHN', 'FHNH', 'FHX', 'FHXH', 'DDFHX', 'FXX', 'FXXH', 'Q', 'SQ', 'SP', 'EV24', 'PG', 'PN', 'PNH', 'PX', 'PXH', 'VVN', 'VVNH', 'VVX', 'VVXH', 'W1', 'W2', 'W3', 'W5', 'W6', 'NG']

r

In [3]:
# Step 2 — NaN handling
SKIP_THRESHOLD = 0.30  # flag a timestep if any station has >30% features missing

# True wherever a value is missing — shape [T, N, F]
nan_mask = np.isnan(data_array)

print(f"Total values : {data_array.size:,}")
print(f"NaN count    : {nan_mask.sum():,}  ({nan_mask.mean():.1%})")

Total values : 17,042,908
NaN count    : 2,847,001  (16.7%)


In [4]:
# For each day and each station: what fraction of the 46 features is missing?
# Shape [T, N]
missing_frac = nan_mask.mean(axis=2)

print(f"missing_frac shape : {missing_frac.shape}")
print(f"  min  : {missing_frac.min():.3f}")
print(f"  max  : {missing_frac.max():.3f}")
print(f"  mean : {missing_frac.mean():.3f}")

missing_frac shape : (10897, 34)
  min  : 0.000
  max  : 1.000
  mean : 0.167


In [5]:
# Flag a timestep if ANY station exceeds the threshold on that day
# Shape [T] — boolean
flagged = (missing_frac > SKIP_THRESHOLD).any(axis=1)

n_flagged = flagged.sum()
print(f"Flagged timesteps : {n_flagged:,} / {T:,}  ({n_flagged/T:.1%})")

# Show first 10 flagged dates
print("\nFirst 10 flagged dates:")
for d in dates[flagged][:10]:
    print(f"  {d.date()}")

Flagged timesteps : 10,897 / 10,897  (100.0%)

First 10 flagged dates:
  1995-01-01
  1995-01-02
  1995-01-03
  1995-01-04
  1995-01-05
  1995-01-06
  1995-01-07
  1995-01-08
  1995-01-09
  1995-01-10


In [6]:
# How many stations per day exceed the threshold?
stations_over_threshold = (missing_frac > SKIP_THRESHOLD).sum(axis=1)

print("How many stations exceed 30% missing per day:")
print(f"  min  : {stations_over_threshold.min()}")
print(f"  max  : {stations_over_threshold.max()}")
print(f"  mean : {stations_over_threshold.mean():.1f}")

# What does the distribution of missing_frac look like?
import numpy as np
percentiles = [50, 75, 90, 95, 99]
print("\nmissing_frac percentiles (across all days and stations):")
for p in percentiles:
    print(f"  {p}th percentile : {np.percentile(missing_frac, p):.3f}")

# Which features are responsible? — NaN rate per feature
missing_per_feature = nan_mask.mean(axis=(0, 1))
print("\nNaN % per feature (sorted worst first):")
order = np.argsort(missing_per_feature)[::-1]
for i in order:
    print(f"  {feature_cols[i]:8s} : {missing_per_feature[i]:.1%}")

How many stations exceed 30% missing per day:
  min  : 7
  max  : 16
  mean : 9.3

missing_frac percentiles (across all days and stations):
  50th percentile : 0.109
  75th percentile : 0.326
  90th percentile : 0.326
  95th percentile : 0.543
  99th percentile : 1.000

NaN % per feature (sorted worst first):
  NG       : 48.5%
  W6       : 45.7%
  W1       : 42.8%
  W3       : 42.8%
  W2       : 42.8%
  W5       : 42.8%
  PN       : 36.4%
  PNH      : 36.4%
  PX       : 36.4%
  PXH      : 36.4%
  PG       : 36.4%
  VVX      : 32.5%
  VVXH     : 32.5%
  VVNH     : 32.5%
  VVN      : 32.5%
  SP       : 7.7%
  SQ       : 7.6%
  EV24     : 7.4%
  Q        : 7.4%
  T10N     : 7.2%
  T10NH    : 7.2%
  FXXH     : 7.0%
  FXX      : 7.0%
  DDVEC    : 6.9%
  FHVEC    : 6.9%
  DDFHX    : 6.9%
  FHXH     : 6.9%
  FHX      : 6.9%
  FHN      : 6.9%
  FHNH     : 6.9%
  FG       : 6.9%
  DR       : 6.9%
  RHXH     : 6.8%
  RHX      : 6.8%
  RH       : 6.8%
  TXH      : 4.6%
  TNH      : 4.6%
  TN    

In [7]:
# Core features we care about for flagging — structurally complete ones
core_features = ['TG', 'TN', 'TX', 'UG', 'RH', 'FG', 'PG', 'Q']

# Get their indices in feature_cols
core_idx = [feature_cols.index(f) for f in core_features]
print("Core feature indices:", core_idx)

# Compute missing fraction using ONLY the core features
# Shape [T, N]
missing_frac_core = nan_mask[:, :, core_idx].mean(axis=2)

print(f"\nmissing_frac_core percentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"  {p}th : {np.percentile(missing_frac_core, p):.3f}")

# How many days get flagged with threshold=0.30 on core features only?
flagged_core = (missing_frac_core > SKIP_THRESHOLD).any(axis=1)
print(f"\nFlagged timesteps : {flagged_core.sum():,} / {T:,}  ({flagged_core.mean():.1%})")

Core feature indices: [0, 1, 3, 7, 13, 17, 31, 27]

missing_frac_core percentiles:
  50th : 0.000
  75th : 0.125
  90th : 0.125
  95th : 0.250
  99th : 1.000

Flagged timesteps : 8,472 / 10,897  (77.7%)


In [8]:
# Check NaN % for each core feature specifically
print("NaN % per core feature:")
for i, name in zip(core_idx, core_features):
    print(f"  {name:6s} : {nan_mask[:, :, i].mean():.1%}")

NaN % per core feature:
  TG     : 4.5%
  TN     : 4.6%
  TX     : 4.6%
  UG     : 4.5%
  RH     : 6.8%
  FG     : 6.9%
  PG     : 36.4%
  Q      : 7.4%


In [9]:
# Remove PG from core features — it is structurally sparse
core_features = ['TG', 'TN', 'TX', 'UG', 'RH', 'FG', 'Q']

core_idx = [feature_cols.index(f) for f in core_features]
print("Core features:", core_features)

# Recompute missing fraction on the cleaned core set
missing_frac_core = nan_mask[:, :, core_idx].mean(axis=2)

print(f"\nmissing_frac_core percentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"  {p}th : {np.percentile(missing_frac_core, p):.3f}")

# How many days get flagged now?
flagged_core = (missing_frac_core > SKIP_THRESHOLD).any(axis=1)
print(f"\nFlagged timesteps : {flagged_core.sum():,} / {T:,}  ({flagged_core.mean():.1%})")

Core features: ['TG', 'TN', 'TX', 'UG', 'RH', 'FG', 'Q']

missing_frac_core percentiles:
  50th : 0.000
  75th : 0.000
  90th : 0.000
  95th : 0.286
  99th : 1.000

Flagged timesteps : 8,461 / 10,897  (77.6%)


In [10]:
# How many stations per day exceed the threshold?
stations_flagged_per_day = (missing_frac_core > SKIP_THRESHOLD).sum(axis=1)

print("Stations exceeding threshold per day:")
print(f"  min  : {stations_flagged_per_day.min()}")
print(f"  max  : {stations_flagged_per_day.max()}")
print(f"  mean : {stations_flagged_per_day.mean():.1f}")

print(f"\nTotal stations : {N}")

# What fraction of days have only 1-2 stations flagged?
for n_thresh in [1, 2, 3, 5]:
    n_days = (stations_flagged_per_day >= n_thresh).sum()
    print(f"  Days where >={n_thresh} station(s) flagged : {n_days:,}  ({n_days/T:.1%})")

Stations exceeding threshold per day:
  min  : 0
  max  : 5
  mean : 1.6

Total stations : 34
  Days where >=1 station(s) flagged : 8,461  (77.6%)
  Days where >=2 station(s) flagged : 5,722  (52.5%)
  Days where >=3 station(s) flagged : 2,151  (19.7%)
  Days where >=5 station(s) flagged : 213  (2.0%)


In [11]:
# Flag a day only if 3 or more stations have missing core features
# Much more reasonable than flagging if even 1 station is missing
MIN_BAD_STATIONS = 3

flagged = (stations_flagged_per_day >= MIN_BAD_STATIONS)

n_flagged = flagged.sum()
print(f"Flagged timesteps : {n_flagged:,} / {T:,}  ({n_flagged/T:.1%})")
print(f"Usable timesteps  : {T - n_flagged:,} / {T:,}  ({(T - n_flagged)/T:.1%})")

# Show first 10 flagged dates
print("\nFirst 10 flagged dates:")
for d in dates[flagged][:10]:
    print(f"  {d.date()}")

Flagged timesteps : 2,151 / 10,897  (19.7%)
Usable timesteps  : 8,746 / 10,897  (80.3%)

First 10 flagged dates:
  1995-01-01
  1995-01-02
  1995-01-03
  1995-01-04
  1995-01-05
  1995-01-06
  1995-01-07
  1995-01-08
  1995-01-09
  1995-01-10


In [12]:
# Fill all NaNs with 0 — after normalisation, 0 = the feature mean
# This is safe neutral imputation, we are not inventing signal
data_filled = np.where(nan_mask, 0.0, data_array).astype(np.float32)

# Verify no NaNs remain
print(f"NaNs before fill : {nan_mask.sum():,}")
print(f"NaNs after fill  : {np.isnan(data_filled).sum():,}")   # must be 0
print(f"Shape unchanged  : {data_filled.shape}")               # must be (10897, 34, 46)

NaNs before fill : 2,847,001
NaNs after fill  : 0
Shape unchanged  : (10897, 34, 46)


In [14]:
import pickle

# Save the results of preprocessing so we can reload without rerunning
preprocessed = {
    "data_filled"       : data_filled,       # [T, N, F] NaNs filled with 0
    "flagged"           : flagged,            # [T] boolean — bad days
    "dates"             : dates,
    "stations"          : stations,
    "feature_cols"      : feature_cols,
    "edge_index"        : prepared["edge_index"],
    "edge_attr"         : prepared["edge_attr"],
    "scalers"           : prepared["scalers"],
    "rh_idx"            : prepared["rh_idx"],
    "K"                 : prepared["K"],
    # Save the config too so we always know what settings produced this file
    "config" : {
        "window_size"      : WINDOW_SIZE,
        "skip_threshold"   : SKIP_THRESHOLD,
        "min_bad_stations" : MIN_BAD_STATIONS,
        "core_features"    : CORE_FEATURES,
    }
}

SAVE_PATH = "/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_preprocessed.pkl"

with open(SAVE_PATH, "wb") as f:
    pickle.dump(preprocessed, f)

print(f"Saved to {SAVE_PATH}")

Saved to /media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_preprocessed.pkl
